In [ ]:
"""
TASK 1: BERT Baseline for Music Tag Understanding — FINAL COMPLETE VERSION
==============================================================================
Run this in a Kaggle Notebook.

BEFORE RUNNING:
1. Add Input: "MusicCaps" (googleai/musiccaps) via the "+" button
2. Settings -> Accelerator -> GPU T4 x2

Dataset: official MusicCaps (musiccaps-public.csv), using the real
"aspect_list" column (musician-written ground-truth tags). Top-50 most
frequent aspects are used as the tag vocabulary.

Guideline compliance included in this version:
- Label leakage prevention: matched aspect phrases are masked out of
  the caption ([MASK]) before being given to BERT as input.
- Macro-F1 / Micro-F1 CURVES vs. training epoch (actual plot, not just
  printed numbers) — required deliverable.
- 5 example predictions — required deliverable.
- B1 majority-class baseline comparison — required minimum baseline.

Copy-paste each "# %% CELL" block into a separate Kaggle notebook cell.
"""

# %% CELL 1 — Install dependencies
!pip install transformers scikit-learn matplotlib -q

# %% CELL 2 — Auto-detect the MusicCaps CSV path
import os

INPUT_ROOT = "/kaggle/input"

def find_path(base_dir, target_name, is_file=False):
    for root, dirs, files_in_dir in os.walk(base_dir):
        if is_file and target_name in files_in_dir:
            return os.path.join(root, target_name)
        if not is_file and target_name in dirs:
            return os.path.join(root, target_name)
    return None

CSV_PATH = find_path(INPUT_ROOT, "musiccaps-public.csv", is_file=True)
print(f"CSV_PATH: {CSV_PATH}")
assert CSV_PATH is not None, "musiccaps-public.csv not found — did you add the MusicCaps dataset via the '+' button?"

WORKING_DIR = "/kaggle/working"

# %% CELL 3 — Imports & Config
import re
import ast
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from collections import Counter

CHECKPOINT_PATH = os.path.join(WORKING_DIR, "task1_checkpoint.pt")
MAX_LEN = 128
BATCH_SIZE = 16
NUM_EPOCHS = 15
LR = 2e-5
TOP_N_TAGS = 50   # how many most-frequent aspects to use as the tag vocabulary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# %% CELL 4 — Load captions and parse aspect_list
df = pd.read_csv(CSV_PATH)
df['aspects_parsed'] = df['aspect_list'].apply(ast.literal_eval)
df['aspects_parsed'] = df['aspects_parsed'].apply(lambda lst: [a.strip().lower() for a in lst])

print(df[['ytid', 'caption', 'aspects_parsed']].head())
print(f"Total captions: {len(df)}")

# %% CELL 5 — Build tag vocabulary from top-N most frequent real aspects
all_aspects = []
for lst in df['aspects_parsed']:
    all_aspects.extend(lst)

counter = Counter(all_aspects)
TAG_NAMES = [tag for tag, _ in counter.most_common(TOP_N_TAGS)]
NUM_TAGS = len(TAG_NAMES)
print(f"Total proxy tags (top {TOP_N_TAGS} real aspects): {NUM_TAGS}")
print(TAG_NAMES)

def build_labels_and_masked_caption(row):
    """
    Build a multi-hot label vector from real aspect_list tags (only the
    ones in our top-N vocabulary), and mask those matched phrases out
    of the caption text to prevent label leakage — without this, BERT
    could just detect the literal keyword instead of learning genuine
    context, inflating F1 to near-100% with no real learning.
    """
    aspects_present = set(row['aspects_parsed'])
    labels = np.zeros(NUM_TAGS, dtype=np.float32)
    masked = row['caption']

    for i, tag in enumerate(TAG_NAMES):
        if tag in aspects_present:
            labels[i] = 1.0
            masked = re.sub(re.escape(tag), "[MASK]", masked, flags=re.IGNORECASE)

    return labels, masked

results = df.apply(build_labels_and_masked_caption, axis=1)
df['labels'] = results.apply(lambda r: r[0])
df['masked_caption'] = results.apply(lambda r: r[1])

no_tag_count = sum(1 for l in df['labels'] if l.sum() == 0)
print(f"\nNo-tag captions (none of the top-{TOP_N_TAGS} aspects matched): {no_tag_count} / {len(df)}")
print("\nExample of masking:")
print(f"Original: {df['caption'].iloc[0]}")
print(f"Masked:   {df['masked_caption'].iloc[0]}")

# %% CELL 6 — Train/Val/Test split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# %% CELL 7 — Dataset class
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class CaptionTagDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=MAX_LEN):
        self.captions = dataframe['masked_caption'].tolist()   # masked, not raw caption
        self.labels = np.stack(dataframe['labels'].tolist())
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.captions)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.captions[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }

train_dataset = CaptionTagDataset(train_df, tokenizer)
val_dataset = CaptionTagDataset(val_df, tokenizer)
test_dataset = CaptionTagDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# %% CELL 8 — BERT Multi-label Classifier Model
class BERTTagClassifier(nn.Module):
    def __init__(self, num_tags=NUM_TAGS):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_tags)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]   # CLS token
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)   # sigmoid is applied inside BCEWithLogitsLoss

# %% CELL 9 — Training setup
model = BERTTagClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()

def train_epoch():
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

@torch.no_grad()
def evaluate(loader, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        preds = (torch.sigmoid(outputs) > threshold).float()
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    return macro_f1, micro_f1, all_preds, all_labels

# %% CELL 10 — Resume from checkpoint (if it exists) + Training loop
# Track Macro-F1 / Micro-F1 per epoch for the required curve plot (CELL 11)
history = {"epoch": [], "train_loss": [], "val_macro_f1": [], "val_micro_f1": []}

start_epoch = 0
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch = ckpt['epoch'] + 1
    history = ckpt.get('history', history)
    print(f"Resumed from epoch {start_epoch}")

for epoch in range(start_epoch, NUM_EPOCHS):
    train_loss = train_epoch()
    val_macro_f1, val_micro_f1, _, _ = evaluate(val_loader)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | Val Macro-F1: {val_macro_f1:.4f} | Val Micro-F1: {val_micro_f1:.4f}")

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["val_macro_f1"].append(val_macro_f1)
    history["val_micro_f1"].append(val_micro_f1)

    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'history': history,
    }, CHECKPOINT_PATH)

# %% CELL 11 — REQUIRED DELIVERABLE: Macro-F1 / Micro-F1 curves vs. training epoch
plt.figure(figsize=(8, 5))
plt.plot(history["epoch"], history["val_macro_f1"], marker='o', label="Val Macro-F1")
plt.plot(history["epoch"], history["val_micro_f1"], marker='s', label="Val Micro-F1")
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("Task 1: Macro-F1 / Micro-F1 vs. Training Epoch")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(WORKING_DIR, "task1_f1_curves.png"), dpi=150)
plt.show()
print("F1 curve plot saved to task1_f1_curves.png")

# %% CELL 12 — Final Test Evaluation
test_macro_f1, test_micro_f1, test_preds, test_labels = evaluate(test_loader)
print(f"\nFinal Test Macro-F1: {test_macro_f1:.4f}")
print(f"Final Test Micro-F1: {test_micro_f1:.4f}")
print("\nPer-tag report:")
print(classification_report(test_labels, test_preds, target_names=TAG_NAMES, zero_division=0))

# %% CELL 13 — REQUIRED DELIVERABLE: 5 example predictions
model.eval()
sample_df = test_df.sample(5, random_state=1)
example_predictions = []
for _, row in sample_df.iterrows():
    encoding = tokenizer(row['masked_caption'], truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model(encoding['input_ids'], encoding['attention_mask'])
        probs = torch.sigmoid(output).cpu().numpy()[0]
    predicted_tags = [TAG_NAMES[i] for i, p in enumerate(probs) if p > 0.5]
    actual_tags = [TAG_NAMES[i] for i, v in enumerate(row['labels']) if v == 1.0]
    print(f"\nOriginal caption: {row['caption'][:100]}...")
    print(f"Model saw (masked): {row['masked_caption'][:100]}...")
    print(f"Actual tags: {actual_tags}")
    print(f"Predicted tags: {predicted_tags}")
    example_predictions.append({
        "caption": row['caption'], "actual_tags": actual_tags, "predicted_tags": predicted_tags
    })

# %% CELL 14 — REQUIRED BASELINE: B1 Majority-Class comparison
# Predicts each tag independently at its overall training-set frequency
# (base rate), thresholded at 0.5 — the simplest required baseline (B1).
train_labels_arr = np.stack(train_df['labels'].tolist())
tag_base_rates = train_labels_arr.mean(axis=0)
b1_pred_vector = (tag_base_rates > 0.5).astype(np.float32)   # same prediction for every sample
b1_preds_repeated = np.tile(b1_pred_vector, (len(test_labels), 1))

b1_macro_f1 = f1_score(test_labels, b1_preds_repeated, average='macro', zero_division=0)
b1_micro_f1 = f1_score(test_labels, b1_preds_repeated, average='micro', zero_division=0)

print(f"\n--- B1 Majority-Class Baseline vs. BERT ---")
print(f"B1 Baseline: Macro-F1 {b1_macro_f1:.4f}, Micro-F1 {b1_micro_f1:.4f}")
print(f"BERT (this model): Macro-F1 {test_macro_f1:.4f}, Micro-F1 {test_micro_f1:.4f}")

# %% CELL 15 — Save all results
results = {
    "test_macro_f1": float(test_macro_f1),
    "test_micro_f1": float(test_micro_f1),
    "b1_baseline_macro_f1": float(b1_macro_f1),
    "b1_baseline_micro_f1": float(b1_micro_f1),
    "tag_names": TAG_NAMES,
    "history": history,
    "example_predictions": example_predictions,
}
RESULTS_PATH = os.path.join(WORKING_DIR, "task1_results.json")
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nCheckpoint saved at: {CHECKPOINT_PATH}")
print(f"Results saved at: {RESULTS_PATH}")
print(f"F1 curve plot saved at: {os.path.join(WORKING_DIR, 'task1_f1_curves.png')}")
print("\nIMPORTANT: click 'Save Version' (top right) now, or these files")
print("will be lost when this session ends.")

In [ ]:
"""
TASK 2: GNN on Music Structure Graphs (FMA-small) — FINAL COMPLETE VERSION
==============================================================================
Run this in a Kaggle Notebook.

BEFORE RUNNING:
1. Add Input: "FMA - Free Music Archive - Small & Medium" (imsparsh/...)
2. Settings -> Accelerator -> GPU T4 x2

Dataset: FMA-small (same as before) — 8,000 tracks, 8 genres.

Guideline compliance included in this version:
- OFFICIAL FMA split (not random) — the assignment requires "Use
  official FMA / MagnaTagATune splits" and "no artist leakage across
  train/test"; FMA's own ('set','split') column satisfies both.
- B1 baseline: majority-class predictor (required minimum baseline).
- B2 baseline: CNN on mel-spectrogram (required deliverable: "Comparison
  vs. CNN baseline on mel-spectrogram").
- GNN (GraphSAGE) on chroma segment-graphs — the main model.
- Accuracy/F1 curve plot vs. training epoch for both GNN and CNN.
- Final 3-way comparison table (B1 vs CNN vs GNN).

Copy-paste each "# %% CELL" block into a separate Kaggle notebook cell.
This is long — expect the full run (preprocessing + GNN + CNN training)
to take a while; preprocessing is CPU-only, so if you want to save GPU
quota, you can run CELL 1-8 (graph preprocessing) with the accelerator
set to None, then switch to GPU before CELL 9 onward.
"""

# %% CELL 1 — Install dependencies
!pip install torch-geometric -q

# %% CELL 2 — Imports & Auto-detect dataset paths
import os
import json
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset as TorchDataset, DataLoader as TorchDataLoader
from torch_geometric.data import Data, Dataset as PyGDataset
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool
from sklearn.metrics import classification_report, f1_score, accuracy_score

INPUT_ROOT = "/kaggle/input"

def find_path(base_dir, target_name, is_file=False):
    for root, dirs, files_in_dir in os.walk(base_dir):
        if is_file and target_name in files_in_dir:
            return os.path.join(root, target_name)
        if not is_file and target_name in dirs:
            return os.path.join(root, target_name)
    return None

def find_audio_root(base_dir):
    """Finds the folder directly containing genre subfolders (000, 001, ...)
    with .mp3 files, handling any nested 'fma_small/fma_small/...' structure."""
    for root, dirs, files_in_dir in os.walk(base_dir):
        if any(f.endswith('.mp3') for f in files_in_dir):
            return os.path.dirname(root)
    return None

FMA_SMALL_CANDIDATE = find_path(INPUT_ROOT, "fma_small")
FMA_AUDIO_DIR = find_audio_root(FMA_SMALL_CANDIDATE) if FMA_SMALL_CANDIDATE else find_audio_root(INPUT_ROOT)
TRACKS_CSV = find_path(INPUT_ROOT, "tracks.csv", is_file=True)

print(f"FMA_AUDIO_DIR: {FMA_AUDIO_DIR}")
print(f"TRACKS_CSV: {TRACKS_CSV}")
assert FMA_AUDIO_DIR is not None, "fma_small mp3 files not found — did you add the dataset via 'Add Input'?"
assert TRACKS_CSV is not None, "tracks.csv not found — did you add the dataset via 'Add Input'?"

WORKING_DIR = "/kaggle/working"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

SEGMENT_DURATION = 5.0
SAMPLE_RATE = 22050
N_CHROMA = 12
SIMILARITY_THRESHOLD = 0.85
N_MELS = 128
FIXED_FRAMES = 640   # ~15s worth of frames at hop_length=512, fixed size for CNN input

# %% CELL 3 — Load metadata (genre labels + OFFICIAL FMA split)
tracks = pd.read_csv(TRACKS_CSV, index_col=0, header=[0, 1])
small_subset = tracks[tracks[('set', 'subset')] == 'small']
genre_labels = small_subset[('track', 'genre_top')].dropna()
official_split = small_subset[('set', 'split')]   # 'training' / 'validation' / 'test'

unique_genres = sorted(genre_labels.unique())
genre_to_idx = {g: i for i, g in enumerate(unique_genres)}
NUM_CLASSES = len(unique_genres)
print(f"Total genres: {NUM_CLASSES}")
print(unique_genres)
print(f"\nOfficial split distribution:\n{official_split.loc[genre_labels.index].value_counts()}")

def track_id_to_path(track_id):
    tid_str = f"{track_id:06d}"
    return os.path.join(FMA_AUDIO_DIR, tid_str[:3], f"{tid_str}.mp3")

# ============================================================
# PART A: GNN on Chroma Segment-Graphs (main model)
# ============================================================

# %% CELL 4 — Audio to Segment Graph builder
def build_segment_graph(audio_path, segment_duration=SEGMENT_DURATION, sr=SAMPLE_RATE):
    """
    Builds a segment-graph from one audio file:
    - Node = chroma feature (mean-pooled, 12-dim) for each 5-second window
    - Edge = temporal adjacency + cosine similarity above threshold
    """
    try:
        y, _ = librosa.load(audio_path, sr=sr, duration=30)
    except Exception:
        return None
    if len(y) < sr:
        return None

    hop_length = 512
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=hop_length)
    frames_per_segment = int(segment_duration * sr / hop_length)
    n_segments = max(1, chroma.shape[1] // frames_per_segment)

    node_features = []
    for i in range(n_segments):
        start = i * frames_per_segment
        end = min(start + frames_per_segment, chroma.shape[1])
        node_features.append(chroma[:, start:end].mean(axis=1))
    if len(node_features) < 2:
        return None
    node_features = np.stack(node_features)

    edges = []
    for i in range(len(node_features)):
        if i + 1 < len(node_features):
            edges.append([i, i + 1]); edges.append([i + 1, i])
        for j in range(i + 1, len(node_features)):
            sim = np.dot(node_features[i], node_features[j]) / (
                np.linalg.norm(node_features[i]) * np.linalg.norm(node_features[j]) + 1e-8)
            if sim > SIMILARITY_THRESHOLD:
                edges.append([i, j]); edges.append([j, i])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    x = torch.tensor(node_features, dtype=torch.float)
    return x, edge_index

# %% CELL 5 — Preprocess all tracks into cached graph files (reuse if available)
GRAPH_CACHE_DIR = os.path.join(WORKING_DIR, "fma_graphs_cache")
EXISTING_GRAPH_CACHE = find_path(INPUT_ROOT, "fma_graphs_cache")
if EXISTING_GRAPH_CACHE is not None and len(os.listdir(EXISTING_GRAPH_CACHE)) > 0:
    GRAPH_CACHE_DIR = EXISTING_GRAPH_CACHE
    print(f"Reusing existing graph cache: {GRAPH_CACHE_DIR} ({len(os.listdir(GRAPH_CACHE_DIR))} files)")
else:
    os.makedirs(GRAPH_CACHE_DIR, exist_ok=True)

def preprocess_all_tracks():
    processed, skipped = 0, 0
    for track_id, genre in genre_labels.items():
        cache_path = os.path.join(GRAPH_CACHE_DIR, f"{track_id}.pt")
        if os.path.exists(cache_path):
            continue
        audio_path = track_id_to_path(track_id)
        if not os.path.exists(audio_path):
            skipped += 1
            continue
        result = build_segment_graph(audio_path)
        if result is None:
            skipped += 1
            continue
        x, edge_index = result
        y = torch.tensor([genre_to_idx[genre]], dtype=torch.long)
        data = Data(x=x, edge_index=edge_index, y=y)
        torch.save(data, cache_path)
        processed += 1
        if processed % 200 == 0:
            print(f"Processed: {processed}, Skipped: {skipped}")
    print(f"DONE. Processed: {processed}, Skipped: {skipped}")

if EXISTING_GRAPH_CACHE is None:
    preprocess_all_tracks()

# %% CELL 6 — Dataset class + OFFICIAL split (not random)
class FMAGraphDataset(PyGDataset):
    def __init__(self, cache_dir):
        super().__init__()
        self.files = [os.path.join(cache_dir, f) for f in os.listdir(cache_dir) if f.endswith('.pt')]
        self.track_ids = [int(os.path.basename(f).replace('.pt', '')) for f in self.files]

    def len(self):
        return len(self.files)

    def get(self, idx):
        return torch.load(self.files[idx], weights_only=False)

full_dataset = FMAGraphDataset(GRAPH_CACHE_DIR)

train_indices, val_indices, test_indices = [], [], []
for i, tid in enumerate(full_dataset.track_ids):
    split_label = official_split.get(tid, "training")
    if split_label == "training":
        train_indices.append(i)
    elif split_label == "validation":
        val_indices.append(i)
    else:
        test_indices.append(i)

train_set = torch.utils.data.Subset(full_dataset, train_indices)
val_set = torch.utils.data.Subset(full_dataset, val_indices)
test_set = torch.utils.data.Subset(full_dataset, test_indices)

train_loader = PyGDataLoader(train_set, batch_size=32, shuffle=True)
val_loader = PyGDataLoader(val_set, batch_size=32)
test_loader = PyGDataLoader(test_set, batch_size=32)
print(f"GNN — Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)} (official FMA split)")

# %% CELL 7 — B1 BASELINE: Majority-class predictor
majority_genre = genre_labels.loc[[full_dataset.track_ids[i] for i in train_indices]].value_counts().idxmax()
test_true_genres = [genre_labels[full_dataset.track_ids[i]] for i in test_indices]
b1_preds = [majority_genre] * len(test_true_genres)

b1_acc = accuracy_score(test_true_genres, b1_preds)
b1_macro_f1 = f1_score(test_true_genres, b1_preds, average='macro', zero_division=0)
print(f"\nB1 Majority-Class Baseline — Test Accuracy: {b1_acc:.4f}, Macro-F1: {b1_macro_f1:.4f}")
print(f"(Majority genre: {majority_genre})")

# %% CELL 8 — GraphSAGE Model
class GraphSAGEClassifier(nn.Module):
    def __init__(self, in_channels=N_CHROMA, hidden_channels=64, num_classes=NUM_CLASSES, num_layers=3):
        super().__init__()
        self.convs = nn.ModuleList([SAGEConv(in_channels, hidden_channels)])
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
        self.classifier = nn.Linear(hidden_channels, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index, batch):
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = self.dropout(x)
        g = global_mean_pool(x, batch)
        return self.classifier(g)

gnn_model = GraphSAGEClassifier().to(device)
gnn_optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.001, weight_decay=5e-4)
gnn_criterion = nn.CrossEntropyLoss()
GNN_CHECKPOINT_PATH = os.path.join(WORKING_DIR, "task2_gnn_checkpoint.pt")

def gnn_train_epoch():
    gnn_model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        gnn_optimizer.zero_grad()
        out = gnn_model(batch.x, batch.edge_index, batch.batch)
        loss = gnn_criterion(out, batch.y)
        loss.backward()
        gnn_optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(train_loader.dataset)

@torch.no_grad()
def gnn_evaluate(loader):
    gnn_model.eval()
    correct, total = 0, 0
    for batch in loader:
        batch = batch.to(device)
        out = gnn_model(batch.x, batch.edge_index, batch.batch)
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
        total += batch.num_graphs
    return correct / total

# %% CELL 9 — GNN Training loop (with history for the required curve plot)
gnn_history = {"epoch": [], "train_loss": [], "val_acc": []}
start_epoch = 0
if os.path.exists(GNN_CHECKPOINT_PATH):
    ckpt = torch.load(GNN_CHECKPOINT_PATH, weights_only=False, map_location=device)
    gnn_model.load_state_dict(ckpt['model_state'])
    gnn_optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch = ckpt['epoch'] + 1
    gnn_history = ckpt.get('history', gnn_history)
    print(f"Resumed from epoch {start_epoch}")

NUM_EPOCHS_GNN = 50
for epoch in range(start_epoch, NUM_EPOCHS_GNN):
    train_loss = gnn_train_epoch()
    val_acc = gnn_evaluate(val_loader)
    print(f"[GNN] Epoch {epoch+1}/{NUM_EPOCHS_GNN} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

    gnn_history["epoch"].append(epoch + 1)
    gnn_history["train_loss"].append(train_loss)
    gnn_history["val_acc"].append(val_acc)

    torch.save({
        'epoch': epoch, 'model_state': gnn_model.state_dict(),
        'optimizer_state': gnn_optimizer.state_dict(), 'history': gnn_history,
    }, GNN_CHECKPOINT_PATH)

# %% CELL 10 — GNN Accuracy curve plot (required visualization)
plt.figure(figsize=(8, 5))
plt.plot(gnn_history["epoch"], gnn_history["val_acc"], marker='o', color='tab:blue')
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Task 2 (GNN): Validation Accuracy vs. Training Epoch")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(WORKING_DIR, "task2_gnn_accuracy_curve.png"), dpi=150)
plt.show()

# %% CELL 11 — GNN Final Test Evaluation
@torch.no_grad()
def gnn_get_predictions(loader):
    gnn_model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        batch = batch.to(device)
        out = gnn_model(batch.x, batch.edge_index, batch.batch)
        pred = out.argmax(dim=1)
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
    return all_preds, all_labels

gnn_test_acc = gnn_evaluate(test_loader)
gnn_preds, gnn_labels = gnn_get_predictions(test_loader)
gnn_macro_f1 = f1_score(gnn_labels, gnn_preds, average='macro')
gnn_micro_f1 = f1_score(gnn_labels, gnn_preds, average='micro')

print(f"\nGNN Final Test Accuracy: {gnn_test_acc:.4f}")
print(f"GNN Macro-F1: {gnn_macro_f1:.4f}")
print(f"GNN Micro-F1: {gnn_micro_f1:.4f}")
print(classification_report(gnn_labels, gnn_preds, target_names=unique_genres))

# ============================================================
# PART B: CNN Baseline (B2) on Mel-Spectrogram
# ============================================================

# %% CELL 12 — Mel-spectrogram extraction (reuse if cached)
MEL_CACHE_DIR = os.path.join(WORKING_DIR, "mel_cache")
EXISTING_MEL_CACHE = find_path(INPUT_ROOT, "mel_cache")
if EXISTING_MEL_CACHE is not None and len(os.listdir(EXISTING_MEL_CACHE)) > 0:
    MEL_CACHE_DIR = EXISTING_MEL_CACHE
    print(f"Reusing existing mel cache: {MEL_CACHE_DIR} ({len(os.listdir(MEL_CACHE_DIR))} files)")
else:
    os.makedirs(MEL_CACHE_DIR, exist_ok=True)

def extract_mel(audio_path):
    try:
        y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, duration=15)
    except Exception:
        return None
    if len(y) < SAMPLE_RATE:
        return None
    mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_mels=N_MELS, hop_length=512)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    if mel_db.shape[1] < FIXED_FRAMES:
        mel_db = np.pad(mel_db, ((0, 0), (0, FIXED_FRAMES - mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :FIXED_FRAMES]
    return mel_db.astype(np.float32)

def preprocess_mel_all():
    processed, skipped = 0, 0
    for track_id, genre in genre_labels.items():
        cache_path = os.path.join(MEL_CACHE_DIR, f"{track_id}.npy")
        if os.path.exists(cache_path):
            continue
        audio_path = track_id_to_path(track_id)
        if not os.path.exists(audio_path):
            skipped += 1
            continue
        mel = extract_mel(audio_path)
        if mel is None:
            skipped += 1
            continue
        np.save(cache_path, mel)
        processed += 1
        if processed % 500 == 0:
            print(f"Processed: {processed}, Skipped: {skipped}")
    print(f"DONE. Processed: {processed}, Skipped: {skipped}")

if EXISTING_MEL_CACHE is None:
    preprocess_mel_all()

# %% CELL 13 — CNN Dataset class using the SAME official split
valid_mel_ids = [tid for tid in genre_labels.index if os.path.exists(os.path.join(MEL_CACHE_DIR, f"{tid}.npy"))]
print(f"Usable tracks for CNN: {len(valid_mel_ids)}")

mel_train_ids = [tid for tid in valid_mel_ids if official_split.get(tid) == "training"]
mel_val_ids = [tid for tid in valid_mel_ids if official_split.get(tid) == "validation"]
mel_test_ids = [tid for tid in valid_mel_ids if official_split.get(tid) == "test"]
print(f"CNN — Train: {len(mel_train_ids)}, Val: {len(mel_val_ids)}, Test: {len(mel_test_ids)} (official FMA split)")

class MelDataset(TorchDataset):
    def __init__(self, track_ids):
        self.track_ids = track_ids

    def __len__(self):
        return len(self.track_ids)

    def __getitem__(self, i):
        tid = self.track_ids[i]
        mel = np.load(os.path.join(MEL_CACHE_DIR, f"{tid}.npy"))
        mel = (mel - mel.mean()) / (mel.std() + 1e-8)
        x = torch.tensor(mel, dtype=torch.float).unsqueeze(0)
        y = torch.tensor(genre_to_idx[genre_labels[tid]], dtype=torch.long)
        return x, y

cnn_train_loader = TorchDataLoader(MelDataset(mel_train_ids), batch_size=32, shuffle=True)
cnn_val_loader = TorchDataLoader(MelDataset(mel_val_ids), batch_size=32)
cnn_test_loader = TorchDataLoader(MelDataset(mel_test_ids), batch_size=32)

# %% CELL 14 — CNN Model + Training
class MelCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 4 * 4, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.conv(x))

cnn_model = MelCNN().to(device)
cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)
cnn_criterion = nn.CrossEntropyLoss()
CNN_CHECKPOINT_PATH = os.path.join(WORKING_DIR, "task2_cnn_checkpoint.pt")

def cnn_train_epoch():
    cnn_model.train()
    total_loss = 0
    for x, y in cnn_train_loader:
        x, y = x.to(device), y.to(device)
        cnn_optimizer.zero_grad()
        out = cnn_model(x)
        loss = cnn_criterion(out, y)
        loss.backward()
        cnn_optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(cnn_train_loader.dataset)

@torch.no_grad()
def cnn_evaluate(loader):
    cnn_model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = cnn_model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

cnn_history = {"epoch": [], "train_loss": [], "val_acc": []}
start_epoch_cnn = 0
if os.path.exists(CNN_CHECKPOINT_PATH):
    ckpt = torch.load(CNN_CHECKPOINT_PATH, weights_only=False, map_location=device)
    cnn_model.load_state_dict(ckpt['model_state'])
    cnn_optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch_cnn = ckpt['epoch'] + 1
    cnn_history = ckpt.get('history', cnn_history)
    print(f"Resumed CNN from epoch {start_epoch_cnn}")

NUM_EPOCHS_CNN = 30
for epoch in range(start_epoch_cnn, NUM_EPOCHS_CNN):
    train_loss = cnn_train_epoch()
    val_acc = cnn_evaluate(cnn_val_loader)
    print(f"[CNN] Epoch {epoch+1}/{NUM_EPOCHS_CNN} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

    cnn_history["epoch"].append(epoch + 1)
    cnn_history["train_loss"].append(train_loss)
    cnn_history["val_acc"].append(val_acc)

    torch.save({
        'epoch': epoch, 'model_state': cnn_model.state_dict(),
        'optimizer_state': cnn_optimizer.state_dict(), 'history': cnn_history,
    }, CNN_CHECKPOINT_PATH)

# %% CELL 15 — CNN Accuracy curve + Final Evaluation
plt.figure(figsize=(8, 5))
plt.plot(cnn_history["epoch"], cnn_history["val_acc"], marker='s', color='tab:orange')
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Task 2 (CNN Baseline): Validation Accuracy vs. Training Epoch")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(WORKING_DIR, "task2_cnn_accuracy_curve.png"), dpi=150)
plt.show()

@torch.no_grad()
def cnn_get_predictions(loader):
    cnn_model.eval()
    all_preds, all_labels = [], []
    for x, y in loader:
        x = x.to(device)
        pred = cnn_model(x).argmax(dim=1)
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(y.numpy())
    return all_preds, all_labels

cnn_test_acc = cnn_evaluate(cnn_test_loader)
cnn_preds, cnn_labels = cnn_get_predictions(cnn_test_loader)
cnn_macro_f1 = f1_score(cnn_labels, cnn_preds, average='macro')
cnn_micro_f1 = f1_score(cnn_labels, cnn_preds, average='micro')

print(f"\nCNN Baseline Final Test Accuracy: {cnn_test_acc:.4f}")
print(f"CNN Baseline Macro-F1: {cnn_macro_f1:.4f}")
print(f"CNN Baseline Micro-F1: {cnn_micro_f1:.4f}")
print(classification_report(cnn_labels, cnn_preds, target_names=unique_genres))

# %% CELL 16 — FINAL 3-WAY COMPARISON TABLE + Save all results
print("\n" + "=" * 60)
print("TASK 2 — FINAL BASELINE COMPARISON (official FMA split)")
print("=" * 60)
print(f"{'Model':<25}{'Accuracy':<12}{'Macro-F1':<12}")
print(f"{'B1 (Majority-class)':<25}{b1_acc:<12.4f}{b1_macro_f1:<12.4f}")
print(f"{'B2 (CNN mel-spectrogram)':<25}{cnn_test_acc:<12.4f}{cnn_macro_f1:<12.4f}")
print(f"{'GNN (GraphSAGE, ours)':<25}{gnn_test_acc:<12.4f}{gnn_macro_f1:<12.4f}")

results = {
    "b1_baseline": {"accuracy": float(b1_acc), "macro_f1": float(b1_macro_f1)},
    "b2_cnn_baseline": {"accuracy": float(cnn_test_acc), "macro_f1": float(cnn_macro_f1), "micro_f1": float(cnn_micro_f1)},
    "gnn_model": {"accuracy": float(gnn_test_acc), "macro_f1": float(gnn_macro_f1), "micro_f1": float(gnn_micro_f1)},
    "genre_names": unique_genres,
    "gnn_history": gnn_history,
    "cnn_history": cnn_history,
    "split_type": "official_fma_split",
}
RESULTS_PATH = os.path.join(WORKING_DIR, "task2_results.json")
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved at: {RESULTS_PATH}")
print("\nIMPORTANT: click 'Save Version' (top right) now, or these files")
print("will be lost when this session ends.")

In [ ]:
"""
TASK 3: GNN-BERT Fusion for Multi-Context Understanding — FIXED / SPEC-COMPLIANT
==================================================================================
Run this in a Kaggle Notebook.

BEFORE RUNNING:
1. Add Input: "MusicCaps" (googleai/musiccaps) — for aspect_list/caption
2. Add Input: "MusicCapsAudio5308" (ayushsharma4045/musiccapsaudio5308) — for audio .wav files
3. (Optional) Add your Task 1 notebook's output as Input, for the BERT warm-start
4. Settings -> Accelerator -> GPU T4 x2

WHAT CHANGED vs. the previous version, and why
------------------------------------------------
The assignment spec (Section 4.3) lists five deliverables for Task 3. The
previous script only satisfied two of them. This version satisfies all five:

  1. End-to-end GNN-BERT fusion model              -> unchanged (Cell 10, 'cross_attention')
  2. Ablation: BERT-only / GNN-only / early-concat  -> NEW (Cell 12): all four
     / cross-attention                                 variants trained & compared
  3. Results (Macro-F1, AUC-PR)                     -> NEW: AUC-PR was missing
                                                        entirely; evaluate() now
                                                        returns it (Cell 11)
  4. t-SNE of z colored by genre AND mood           -> FIXED: previous version
                                                        colored by "top tag" only,
                                                        not genre/mood specifically
                                                        (Cell 14)
  5. 3 case studies showing graph paths +           -> FIXED: previous version
     caption/lyric alignment                           printed caption + audio
                                                        path only, no graph
                                                        structure or predictions
                                                        (Cell 15)

DATASET NOTE: the spec's suggested pairing for Task 3 is FMA-medium or
MagnaTagATune. This script keeps MusicCaps instead, because Task 1's BERT
checkpoint, the cached segment graphs, and Task 4's contrastive retrieval
are all already built on MusicCaps — switching now would break checkpoint
and cache reuse across the whole pipeline. If your instructor requires the
literal FMA-medium/MagnaTagATune pairing, that needs a separate data-loading
cell (different CSV, different label schema) — flag it and I'll write that
version.

MusicCaps has no explicit genre/mood columns, so "genre" and "mood" for the
t-SNE plots are derived from the tag vocabulary itself via keyword rules
(Cell 4b) — e.g. "rock", "pop", "instrumental" -> genre bucket;
"energetic", "happy", "romantic" -> mood bucket. This is a documented
approximation, not ground-truth genre/mood labels.

Copy-paste each "# %% CELL" block into a separate Kaggle notebook cell.
"""

# %% CELL 1 — Install dependencies
!pip install torch-geometric transformers scikit-learn matplotlib -q

# %% CELL 2 — Auto-detect dataset paths
import os

INPUT_ROOT = "/kaggle/input"

def find_path(base_dir, target_name, is_file=False):
    for root, dirs, files_in_dir in os.walk(base_dir):
        if is_file and target_name in files_in_dir:
            return os.path.join(root, target_name)
        if not is_file and target_name in dirs:
            return os.path.join(root, target_name)
    return None

CSV_PATH = find_path(INPUT_ROOT, "musiccaps-public.csv", is_file=True)

def find_wav_root(base_dir):
    for root, dirs, files_in_dir in os.walk(base_dir):
        if any(f.endswith('.wav') for f in files_in_dir):
            return root
    return None

AUDIO_FILES_CANDIDATE = find_path(INPUT_ROOT, "audioFiles")
AUDIO_DIR = find_wav_root(AUDIO_FILES_CANDIDATE) if AUDIO_FILES_CANDIDATE else find_wav_root(INPUT_ROOT)

print(f"CSV_PATH: {CSV_PATH}")
print(f"AUDIO_DIR: {AUDIO_DIR}")
assert CSV_PATH is not None, "musiccaps-public.csv not found — add the MusicCaps dataset"
assert AUDIO_DIR is not None, "No .wav files found — add the MusicCapsAudio5308 dataset"

WORKING_DIR = "/kaggle/working"

TASK1_CHECKPOINT = find_path(INPUT_ROOT, "task1_checkpoint.pt", is_file=True)
print(f"Task 1 checkpoint (optional warm-start): {TASK1_CHECKPOINT}")

# %% CELL 3 — Imports & Config
import re
import ast
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import matplotlib.pyplot as plt
from torch.utils.data import Dataset as TorchDataset, DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool
from transformers import BertTokenizer, BertModel
from sklearn.metrics import f1_score, average_precision_score, classification_report
from collections import Counter

MAX_LEN = 128
BATCH_SIZE = 8
NUM_EPOCHS = 20          # epochs for the primary (cross-attention) model
ABLATION_EPOCHS = 10     # shorter budget for the 3 comparison variants
TOP_N_TAGS = 50
SEGMENT_DURATION = 2.0
SAMPLE_RATE = 22050
N_CHROMA = 12
SIMILARITY_THRESHOLD = 0.85

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# %% CELL 4 — Load captions, parse aspect_list, build tag vocabulary
df = pd.read_csv(CSV_PATH)
df['aspects_parsed'] = df['aspect_list'].apply(ast.literal_eval)
df['aspects_parsed'] = df['aspects_parsed'].apply(lambda lst: [a.strip().lower() for a in lst])

all_aspects = []
for lst in df['aspects_parsed']:
    all_aspects.extend(lst)
counter = Counter(all_aspects)
TAG_NAMES = [tag for tag, _ in counter.most_common(TOP_N_TAGS)]
NUM_TAGS = len(TAG_NAMES)
print(f"Total tags: {NUM_TAGS}")

def build_labels_and_masked_caption(row):
    aspects_present = set(row['aspects_parsed'])
    labels = np.zeros(NUM_TAGS, dtype=np.float32)
    masked = row['caption']
    for i, tag in enumerate(TAG_NAMES):
        if tag in aspects_present:
            labels[i] = 1.0
            masked = re.sub(re.escape(tag), "[MASK]", masked, flags=re.IGNORECASE)
    return labels, masked

results = df.apply(build_labels_and_masked_caption, axis=1)
df['labels'] = results.apply(lambda r: r[0])
df['masked_caption'] = results.apply(lambda r: r[1])

# %% CELL 4b — NEW: derive pseudo genre/mood buckets from the tag vocabulary
# MusicCaps has no explicit genre/mood columns. We bucket each of the top-50
# tags into a category via keyword rules, so the t-SNE plots required by the
# spec (colored by genre AND mood) have something meaningful to color by.
MOOD_KEYWORDS = ['emotional', 'passionate', 'energetic', 'groovy', 'happy', 'spirited',
                  'romantic', 'exciting', 'upbeat', 'mellow', 'fun', 'cheerful',
                  'easygoing', 'youthful', 'sad', 'calm', 'relaxing', 'dark', 'intense']
GENRE_KEYWORDS = ['rock', 'pop', 'jazz', 'hip-hop', 'hip hop', 'electro', 'electronic',
                   'folk', 'classical', 'country', 'reggae', 'blues', 'metal', 'punk',
                   'funk', 'disco', 'afrobeat', 'latin', 'instrumental']
TEMPO_KEYWORDS = ['tempo', 'uptempo']
VOCAL_KEYWORDS = ['vocal', 'voice', 'singer', 'singing']
INSTRUMENT_KEYWORDS = ['guitar', 'drum', 'bass', 'piano', 'kick', 'snare', 'hat',
                        'percussion', 'keyboard', 'synth', 'strings', 'brass', 'violin']
PRODUCTION_KEYWORDS = ['quality', 'noisy', 'amateur', 'live performance', 'mono', 'loud', 'studio']

def categorize_tag(tag):
    t = tag.lower()
    if any(k in t for k in MOOD_KEYWORDS):
        return 'mood'
    if any(k in t for k in TEMPO_KEYWORDS):
        return 'tempo'
    if any(k in t for k in VOCAL_KEYWORDS):
        return 'vocal'
    if any(k in t for k in INSTRUMENT_KEYWORDS):
        return 'instrument'
    if any(k in t for k in PRODUCTION_KEYWORDS):
        return 'production'
    if any(k in t for k in GENRE_KEYWORDS):
        return 'genre'
    return 'other'

TAG_CATEGORY = {tag: categorize_tag(tag) for tag in TAG_NAMES}
MOOD_TAG_IDX = [i for i, t in enumerate(TAG_NAMES) if TAG_CATEGORY[t] == 'mood']
GENRE_TAG_IDX = [i for i, t in enumerate(TAG_NAMES) if TAG_CATEGORY[t] == 'genre']
print(f"Mood-bucket tags ({len(MOOD_TAG_IDX)}): {[TAG_NAMES[i] for i in MOOD_TAG_IDX]}")
print(f"Genre-bucket tags ({len(GENRE_TAG_IDX)}): {[TAG_NAMES[i] for i in GENRE_TAG_IDX]}")

def dominant_label(label_vec, idx_list, fallback='none'):
    present = [TAG_NAMES[i] for i in idx_list if label_vec[i] == 1.0]
    return present[0] if present else fallback

# %% CELL 5 — Match ytid to actual audio files (robust matching)
def normalize_id(s):
    s = s[:-4] if s.endswith('.wav') else s
    if s.startswith('-') or s.startswith('_'):
        s = s[1:]
    return s

wav_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')]
normalized_lookup = {normalize_id(f): f for f in wav_files}

def find_audio_file(ytid):
    key = normalize_id(ytid)
    return os.path.join(AUDIO_DIR, normalized_lookup[key]) if key in normalized_lookup else None

df['audio_path'] = df['ytid'].apply(find_audio_file)
df_matched = df[df['audio_path'].notna()].reset_index(drop=True)
print(f"Matched {len(df_matched)} / {len(df)} captions to audio files")
assert len(df_matched) > 0, "No matches found — check AUDIO_DIR contents and ytid format"

# %% CELL 6 — Audio to Segment Graph builder (same approach as Task 2, chroma-only)
def build_segment_graph(audio_path, segment_duration=SEGMENT_DURATION, sr=SAMPLE_RATE):
    try:
        y, _ = librosa.load(audio_path, sr=sr, duration=10)
    except Exception:
        return None
    if len(y) < sr:
        return None

    hop_length = 512
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=hop_length)
    frames_per_segment = int(segment_duration * sr / hop_length)
    n_segments = max(1, chroma.shape[1] // frames_per_segment)

    node_features = []
    for i in range(n_segments):
        start = i * frames_per_segment
        end = min(start + frames_per_segment, chroma.shape[1])
        node_features.append(chroma[:, start:end].mean(axis=1))

    if len(node_features) < 2:
        return None
    node_features = np.stack(node_features)

    edges = []
    for i in range(len(node_features)):
        if i + 1 < len(node_features):
            edges.append([i, i + 1]); edges.append([i + 1, i])
        for j in range(i + 1, len(node_features)):
            sim = np.dot(node_features[i], node_features[j]) / (
                np.linalg.norm(node_features[i]) * np.linalg.norm(node_features[j]) + 1e-8)
            if sim > SIMILARITY_THRESHOLD:
                edges.append([i, j]); edges.append([j, i])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    x = torch.tensor(node_features, dtype=torch.float)
    return x, edge_index

# %% CELL 7 — Preprocess into graph cache (skip if already cached from a previous session)
GRAPH_CACHE_DIR = os.path.join(WORKING_DIR, "task3_graphs_cache")
EXISTING_CACHE = find_path(INPUT_ROOT, "task3_graphs_cache")
if EXISTING_CACHE is not None and len(os.listdir(EXISTING_CACHE)) > 0:
    GRAPH_CACHE_DIR = EXISTING_CACHE
    print(f"Reusing existing graph cache: {GRAPH_CACHE_DIR}")
else:
    os.makedirs(GRAPH_CACHE_DIR, exist_ok=True)

def preprocess_all():
    processed, skipped = 0, 0
    for idx, row in df_matched.iterrows():
        cache_path = os.path.join(GRAPH_CACHE_DIR, f"{idx}.pt")
        if os.path.exists(cache_path):
            continue
        result = build_segment_graph(row['audio_path'])
        if result is None:
            skipped += 1
            continue
        x, edge_index = result
        torch.save({'x': x, 'edge_index': edge_index}, cache_path)
        processed += 1
        if processed % 500 == 0:
            print(f"Processed: {processed}, Skipped: {skipped}")
    print(f"DONE. Processed: {processed}, Skipped: {skipped}")

if EXISTING_CACHE is None:
    preprocess_all()

valid_indices = [i for i in range(len(df_matched)) if os.path.exists(os.path.join(GRAPH_CACHE_DIR, f"{i}.pt"))]
df_final = df_matched.iloc[valid_indices].reset_index(drop=True)
graph_files = [os.path.join(GRAPH_CACHE_DIR, f"{i}.pt") for i in valid_indices]
print(f"Final usable samples: {len(df_final)}")

# %% CELL 8 — Train/Val/Test split
from sklearn.model_selection import train_test_split

indices = list(range(len(df_final)))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# %% CELL 9 — Dataset class (graph + tokenized masked caption + labels)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class FusionDataset(TorchDataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        row = df_final.iloc[idx]
        graph_data = torch.load(graph_files[idx], weights_only=False)
        encoding = tokenizer(
            row['masked_caption'], truncation=True, padding='max_length',
            max_length=MAX_LEN, return_tensors='pt'
        )
        return {
            'x': graph_data['x'],
            'edge_index': graph_data['edge_index'],
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(row['labels'], dtype=torch.float)
        }

def collate_fn(batch):
    x_list, edge_index_list, batch_idx = [], [], []
    node_offset = 0
    for i, item in enumerate(batch):
        x_list.append(item['x'])
        edge_index_list.append(item['edge_index'] + node_offset)
        batch_idx.extend([i] * item['x'].shape[0])
        node_offset += item['x'].shape[0]

    return {
        'x': torch.cat(x_list, dim=0),
        'edge_index': torch.cat(edge_index_list, dim=1),
        'batch': torch.tensor(batch_idx, dtype=torch.long),
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels': torch.stack([b['labels'] for b in batch]),
    }

train_loader = DataLoader(FusionDataset(train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(FusionDataset(val_idx), batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(FusionDataset(test_idx), batch_size=BATCH_SIZE, collate_fn=collate_fn)

# %% CELL 10 — NEW: Unified model supporting all 4 ablation variants
class GNNEncoder(nn.Module):
    """Same architecture as Task 2's GraphSAGE, outputs a graph embedding."""
    def __init__(self, in_channels=N_CHROMA, hidden_channels=64, num_layers=3):
        super().__init__()
        self.convs = nn.ModuleList([SAGEConv(in_channels, hidden_channels)])
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index, batch):
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = self.dropout(x)
        return global_mean_pool(x, batch)


class UnifiedFusionModel(nn.Module):
    """
    fusion_type in {'bert_only', 'gnn_only', 'early_concat', 'cross_attention'}.
    Required by spec Section 4.3 deliverable #2: ablation across all four.
    """
    def __init__(self, fusion_type, num_tags=NUM_TAGS, gnn_hidden=64, bert_hidden=768):
        super().__init__()
        assert fusion_type in ('bert_only', 'gnn_only', 'early_concat', 'cross_attention')
        self.fusion_type = fusion_type

        if fusion_type != 'bert_only':
            self.gnn = GNNEncoder(hidden_channels=gnn_hidden)
        if fusion_type != 'gnn_only':
            self.bert = BertModel.from_pretrained('bert-base-uncased')

        if fusion_type == 'cross_attention':
            self.query_proj = nn.Linear(gnn_hidden, bert_hidden)
            self.attn = nn.MultiheadAttention(embed_dim=bert_hidden, num_heads=8, batch_first=True)
            classifier_in = gnn_hidden + bert_hidden
        elif fusion_type == 'early_concat':
            classifier_in = gnn_hidden + bert_hidden
        elif fusion_type == 'gnn_only':
            classifier_in = gnn_hidden
        else:  # bert_only
            classifier_in = bert_hidden

        self.classifier = nn.Sequential(
            nn.Linear(classifier_in, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_tags)
        )

    def forward(self, x, edge_index, batch, input_ids, attention_mask):
        if self.fusion_type == 'bert_only':
            t = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
            return self.classifier(t)

        if self.fusion_type == 'gnn_only':
            g = self.gnn(x, edge_index, batch)
            return self.classifier(g)

        g = self.gnn(x, edge_index, batch)
        Htext = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

        if self.fusion_type == 'early_concat':
            t_cls = Htext[:, 0, :]
            z = torch.cat([g, t_cls], dim=1)
            return self.classifier(z)

        # cross_attention
        q = self.query_proj(g).unsqueeze(1)
        attended, _ = self.attn(q, Htext, Htext, key_padding_mask=(attention_mask == 0))
        z = torch.cat([g, attended.squeeze(1)], dim=1)
        return self.classifier(z)

    def get_fused_embedding(self, x, edge_index, batch, input_ids, attention_mask):
        """Only defined for cross_attention — used for the t-SNE cell."""
        assert self.fusion_type == 'cross_attention'
        g = self.gnn(x, edge_index, batch)
        Htext = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        q = self.query_proj(g).unsqueeze(1)
        attended, _ = self.attn(q, Htext, Htext, key_padding_mask=(attention_mask == 0))
        return torch.cat([g, attended.squeeze(1)], dim=1)


def maybe_warm_start_bert(model):
    if TASK1_CHECKPOINT is not None and hasattr(model, 'bert'):
        ckpt = torch.load(TASK1_CHECKPOINT, weights_only=False)
        bert_state = {k.replace('bert.', ''): v for k, v in ckpt['model_state'].items() if k.startswith('bert.')}
        model.bert.load_state_dict(bert_state, strict=False)
        return True
    return False

# %% CELL 11 — NEW: training/eval helpers, now with AUC-PR
criterion = nn.BCEWithLogitsLoss()

def train_epoch(model, optimizer, loader):
    model.train()
    total_loss = 0
    for batch in loader:
        x = batch['x'].to(device)
        edge_index = batch['edge_index'].to(device)
        b_idx = batch['batch'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(x, edge_index, b_idx, input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    for batch in loader:
        x = batch['x'].to(device)
        edge_index = batch['edge_index'].to(device)
        b_idx = batch['batch'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(x, edge_index, b_idx, input_ids, attention_mask)
        probs = torch.sigmoid(outputs)
        preds = (probs > threshold).float()
        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)

    # AUC-PR (spec Section 6): mean average precision over tags that have at
    # least one positive example in this split (undefined otherwise).
    valid_tags = [k for k in range(all_labels.shape[1]) if all_labels[:, k].sum() > 0]
    if valid_tags:
        auc_pr = average_precision_score(all_labels[:, valid_tags], all_probs[:, valid_tags], average='macro')
    else:
        auc_pr = float('nan')

    return {'macro_f1': macro_f1, 'micro_f1': micro_f1, 'auc_pr': auc_pr}, all_probs, all_preds, all_labels

# %% CELL 12 — NEW: Ablation study (spec deliverable #2)
# Trains all four fusion variants and compares them. The primary
# 'cross_attention' model gets the full NUM_EPOCHS budget with
# checkpoint/resume support; the other three get ABLATION_EPOCHS since
# they exist purely for comparison, not as the final deliverable model.
FUSION_TYPES = ['bert_only', 'gnn_only', 'early_concat', 'cross_attention']
ablation_results = {}
models = {}

for fusion_type in FUSION_TYPES:
    print(f"\n{'='*60}\nTraining variant: {fusion_type}\n{'='*60}")
    model = UnifiedFusionModel(fusion_type).to(device)
    warm_started = maybe_warm_start_bert(model)
    if warm_started:
        print("Warm-started BERT weights from Task 1 checkpoint.")

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    epochs = NUM_EPOCHS if fusion_type == 'cross_attention' else ABLATION_EPOCHS
    ckpt_path = os.path.join(WORKING_DIR, f"task3_{fusion_type}_checkpoint.pt")

    start_epoch = 0
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed {fusion_type} from epoch {start_epoch}")

    for epoch in range(start_epoch, epochs):
        train_loss = train_epoch(model, optimizer, train_loader)
        val_metrics, _, _, _ = evaluate(model, val_loader)
        print(f"[{fusion_type}] Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | "
              f"Val Macro-F1: {val_metrics['macro_f1']:.4f} | Val AUC-PR: {val_metrics['auc_pr']:.4f}")
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict()}, ckpt_path)

    test_metrics, _, _, _ = evaluate(model, test_loader)
    ablation_results[fusion_type] = test_metrics
    models[fusion_type] = model
    print(f"[{fusion_type}] TEST — Macro-F1: {test_metrics['macro_f1']:.4f} | "
          f"Micro-F1: {test_metrics['micro_f1']:.4f} | AUC-PR: {test_metrics['auc_pr']:.4f}")

main_model = models['cross_attention']  # the primary deliverable model

# %% CELL 13 — NEW: Ablation comparison table (spec Table 3 style)
print(f"\n{'='*70}\nTASK 3 — ABLATION COMPARISON (spec Section 4.3 / Table 3)\n{'='*70}")
print(f"{'Model':<20}{'Macro-F1':<12}{'Micro-F1':<12}{'AUC-PR':<12}")
label_map = {'bert_only': 'BERT-only', 'gnn_only': 'GNN-only',
             'early_concat': 'Early-concat', 'cross_attention': 'Cross-attention (ours)'}
for ft in FUSION_TYPES:
    m = ablation_results[ft]
    print(f"{label_map[ft]:<20}{m['macro_f1']:<12.4f}{m['micro_f1']:<12.4f}{m['auc_pr']:<12.4f}")

# %% CELL 14 — FIXED: t-SNE colored by genre AND by mood (spec deliverable #4)
from sklearn.manifold import TSNE

@torch.no_grad()
def get_fused_embeddings_and_labels(model, loader):
    model.eval()
    embeddings, mood_labels, genre_labels = [], [], []
    for batch in loader:
        x = batch['x'].to(device)
        edge_index = batch['edge_index'].to(device)
        b_idx = batch['batch'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].numpy()

        z = model.get_fused_embedding(x, edge_index, b_idx, input_ids, attention_mask)
        embeddings.append(z.cpu().numpy())
        for lbl in labels:
            mood_labels.append(dominant_label(lbl, MOOD_TAG_IDX))
            genre_labels.append(dominant_label(lbl, GENRE_TAG_IDX))
    return np.concatenate(embeddings), mood_labels, genre_labels

embeddings, mood_labels, genre_labels = get_fused_embeddings_and_labels(main_model, test_loader)
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings) - 1))
proj = tsne.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, group_labels, title in [(axes[0], genre_labels, "colored by genre"),
                                  (axes[1], mood_labels, "colored by mood")]:
    unique_vals = list(set(group_labels))[:15]  # cap legend at 15 for readability
    for val in unique_vals:
        mask = [g == val for g in group_labels]
        ax.scatter(proj[mask, 0], proj[mask, 1], label=val, alpha=0.6, s=15)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
    ax.set_title(f"t-SNE of Fused GNN-BERT Embeddings, {title}")
plt.tight_layout()
plt.savefig(os.path.join(WORKING_DIR, "task3_tsne_genre_mood.png"), dpi=150)
plt.show()

# %% CELL 15 — FIXED: case studies with graph paths + caption/lyric alignment
# spec deliverable #5 explicitly asks for "graph paths + caption/lyric
# alignment" — the previous version only printed the audio path and caption.
# This version prints: segment-graph structure (nodes + adjacency), which
# words were masked out for label-leakage prevention (the alignment between
# graph segments and text is implicit in "same clip, same tags"), and the
# model's actual top-5 predicted tags next to the ground truth so the
# alignment between graph-driven prediction and text is visible.
main_model.eval()
np.random.seed(7)
sample_indices = np.random.choice(len(test_idx), min(3, len(test_idx)), replace=False)

for i in sample_indices:
    idx = test_idx[i]
    row = df_final.iloc[idx]
    graph_data = torch.load(graph_files[idx], weights_only=False)
    x, edge_index = graph_data['x'], graph_data['edge_index']

    adjacency = {}
    for e in range(edge_index.shape[1]):
        src, dst = edge_index[0, e].item(), edge_index[1, e].item()
        adjacency.setdefault(src, set()).add(dst)
    adjacency = {k: sorted(v) for k, v in adjacency.items()}

    encoding = tokenizer(row['masked_caption'], truncation=True, padding='max_length',
                          max_length=MAX_LEN, return_tensors='pt')
    with torch.no_grad():
        logits = main_model(
            x.to(device), edge_index.to(device),
            torch.zeros(x.shape[0], dtype=torch.long).to(device),
            encoding['input_ids'].to(device), encoding['attention_mask'].to(device)
        )
        probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
    top5_idx = np.argsort(probs)[::-1][:5]
    predicted = [(TAG_NAMES[j], round(float(probs[j]), 3)) for j in top5_idx]
    actual = [TAG_NAMES[j] for j, v in enumerate(row['labels']) if v == 1.0]

    print(f"\n--- Case Study (test idx {idx}) ---")
    print(f"Audio file: {row['audio_path']}")
    print(f"Segment graph: {x.shape[0]} nodes (~{SEGMENT_DURATION}s each), "
          f"{edge_index.shape[1]} directed edges")
    print(f"Graph adjacency (segment -> connected segments): {adjacency}")
    print(f"Original caption: {row['caption'][:150]}...")
    print(f"Masked caption (aspect keywords -> [MASK] to prevent leakage): {row['masked_caption'][:150]}...")
    print(f"Actual tags: {actual}")
    print(f"Model top-5 predicted tags (tag, probability): {predicted}")

# %% CELL 16 — Save results
RESULTS_PATH = os.path.join(WORKING_DIR, "task3_results.json")
results = {
    "ablation_results": ablation_results,
    "primary_model": "cross_attention",
    "tag_names": TAG_NAMES,
    "tag_categories": TAG_CATEGORY,
    "num_samples": len(df_final),
}
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

torch.save({'model_state': main_model.state_dict()}, os.path.join(WORKING_DIR, "task3_checkpoint.pt"))

print(f"Results saved at: {RESULTS_PATH}")
print("\nIMPORTANT: click 'Save Version' (top right) now, or these files")
print("will be lost when this session ends.")

In [1]:
"""
TASK 4: Contrastive GNN-BERT for MusicCaps Cross-Modal Retrieval — FIXED / SPEC-COMPLIANT
============================================================================================
Run this in a Kaggle Notebook (can be the same notebook as Task 3, or a
separate one — this script is written to run as a SEPARATE notebook that
pulls Task 3's outputs in via Kaggle Input, since that's how you're set up).

BEFORE RUNNING:
1. Add Input: "MusicCaps" (googleai/musiccaps)
2. Add Input: "MusicCapsAudio5308" (ayushsharma4045/musiccapsaudio5308)
3. Add Input: your Task 3 notebook's saved output (the committed version that
   has task3_checkpoint.pt, task3_graphs_cache/, task3_results.json in
   /kaggle/working). Kaggle -> "+ Add Input" -> "Your Work" -> select that
   notebook's latest version. This is what lets this script SKIP graph
   preprocessing and reuse Task 3's trained GNN + BERT weights instead of
   recomputing anything.
4. Settings -> Accelerator -> GPU T4 x2

WHAT CHANGED vs. the previous version, and why
------------------------------------------------
Spec Section 4.4 lists four deliverables for Task 4. The previous script
only satisfied three of them:

  1. Dual-encoder GNN-BERT with contrastive training   -> unchanged (Cell 10-13)
  2. Retrieval evaluation table (R@1/5/10, both dirs)   -> unchanged, now
                                                            printed as a clean
                                                            table (Cell 14)
  3. 10 qualitative retrieval examples                  -> unchanged (Cell 15)
  4. Zero-shot tag prediction from captions vs.          -> NEW (Cell 16):
     Task 3 supervised model                                this was missing
                                                              entirely before

Also added: a human-evaluation template (Cell 17), because spec Section 6
("Human evaluation (Task 4). Minimum 5 listeners rate whether retrieved
clip matches caption on scale [1,5]") can't be automated — this exports a
CSV with the queries + top-3 matches + blank rating columns for 5 listeners
to fill in by hand, so you have something to attach to the report.

Copy-paste each "# %% CELL" block into a separate Kaggle notebook cell.
"""

# %% CELL 1 — Install dependencies
!pip install torch-geometric transformers scikit-learn -q

# %% CELL 2 — Auto-detect dataset + Task 3 output paths
import os

INPUT_ROOT = "/kaggle/input"

def find_path(base_dir, target_name, is_file=False):
    for root, dirs, files_in_dir in os.walk(base_dir):
        if is_file and target_name in files_in_dir:
            return os.path.join(root, target_name)
        if not is_file and target_name in dirs:
            return os.path.join(root, target_name)
    return None

def find_wav_root(base_dir):
    for root, dirs, files_in_dir in os.walk(base_dir):
        if any(f.endswith('.wav') for f in files_in_dir):
            return root
    return None

CSV_PATH = find_path(INPUT_ROOT, "musiccaps-public.csv", is_file=True)
AUDIO_FILES_CANDIDATE = find_path(INPUT_ROOT, "audioFiles")
AUDIO_DIR = find_wav_root(AUDIO_FILES_CANDIDATE) if AUDIO_FILES_CANDIDATE else find_wav_root(INPUT_ROOT)

print(f"CSV_PATH: {CSV_PATH}")
print(f"AUDIO_DIR: {AUDIO_DIR}")
assert CSV_PATH is not None, "musiccaps-public.csv not found — add the MusicCaps dataset"
assert AUDIO_DIR is not None, "No .wav files found — add the MusicCapsAudio5308 dataset"

WORKING_DIR = "/kaggle/working"

# Task 3 artifacts — all pulled from Input, none of this gets recomputed
TASK3_CHECKPOINT = find_path(INPUT_ROOT, "task3_checkpoint.pt", is_file=True)
TASK3_GRAPH_CACHE = find_path(INPUT_ROOT, "task3_graphs_cache")
TASK3_RESULTS = find_path(INPUT_ROOT, "task3_results.json", is_file=True)
print(f"Task 3 checkpoint (warm-start GNN+BERT): {TASK3_CHECKPOINT}")
print(f"Task 3 graph cache (skip preprocessing): {TASK3_GRAPH_CACHE}")
print(f"Task 3 results.json (tag vocab + supervised scores): {TASK3_RESULTS}")

# %% CELL 3 — Imports & Config
import re
import ast
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
from torch.utils.data import Dataset as TorchDataset, DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool
from transformers import BertTokenizer, BertModel
from sklearn.metrics import f1_score, average_precision_score
from collections import Counter

MAX_LEN = 128
BATCH_SIZE = 32
NUM_EPOCHS = 20
LR = 2e-5
SEGMENT_DURATION = 2.0
SAMPLE_RATE = 22050
N_CHROMA = 12
SIMILARITY_THRESHOLD = 0.85
TEMPERATURE = 0.07
EMBED_DIM = 256
TOP_N_TAGS = 50  # only used as a fallback if task3_results.json isn't found

CHECKPOINT_PATH = os.path.join(WORKING_DIR, "task4_checkpoint.pt")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# %% CELL 4 — Load captions + tag vocabulary (reuse Task 3's exact vocabulary)
df = pd.read_csv(CSV_PATH)
df['aspects_parsed'] = df['aspect_list'].apply(ast.literal_eval)
df['aspects_parsed'] = df['aspects_parsed'].apply(lambda lst: [a.strip().lower() for a in lst])

if TASK3_RESULTS is not None:
    with open(TASK3_RESULTS) as f:
        task3_results = json.load(f)
    TAG_NAMES = task3_results['tag_names']
    print(f"Loaded {len(TAG_NAMES)} tags from Task 3's results.json (same vocabulary — required "
          f"for the zero-shot-vs-supervised comparison in Cell 16 to be apples-to-apples).")
else:
    # Fallback: recompute the same way Task 3 did. Only used if task3_results.json
    # wasn't added as an Input — the zero-shot-vs-Task-3 comparison in Cell 16 will
    # be skipped in that case since there's no supervised score to compare against.
    print("WARNING: task3_results.json not found — recomputing tag vocabulary from "
          "scratch. This should match Task 3's vocabulary exactly since it's the same "
          "deterministic top-N-by-frequency logic, but the Cell 16 comparison needs "
          "Task 3's actual saved scores, so add that file as an Input if you want it.")
    all_aspects = []
    for lst in df['aspects_parsed']:
        all_aspects.extend(lst)
    counter = Counter(all_aspects)
    TAG_NAMES = [tag for tag, _ in counter.most_common(TOP_N_TAGS)]
    task3_results = None

NUM_TAGS = len(TAG_NAMES)

def build_labels(row):
    aspects_present = set(row['aspects_parsed'])
    labels = np.zeros(NUM_TAGS, dtype=np.float32)
    for i, tag in enumerate(TAG_NAMES):
        if tag in aspects_present:
            labels[i] = 1.0
    return labels

df['labels'] = df.apply(build_labels, axis=1)

# %% CELL 5 — Match ytid to audio files (same robust logic as Task 3)
def normalize_id(s):
    s = s[:-4] if s.endswith('.wav') else s
    if s.startswith('-') or s.startswith('_'):
        s = s[1:]
    return s

wav_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')]
normalized_lookup = {normalize_id(f): f for f in wav_files}

def find_audio_file(ytid):
    key = normalize_id(ytid)
    return os.path.join(AUDIO_DIR, normalized_lookup[key]) if key in normalized_lookup else None

df['audio_path'] = df['ytid'].apply(find_audio_file)
df_matched = df[df['audio_path'].notna()].reset_index(drop=True)
print(f"Matched {len(df_matched)} / {len(df)} captions to audio files")
assert len(df_matched) > 0, "No matches found"

# %% CELL 6 — Audio to Segment Graph builder (identical to Task 3, for cache compatibility)
# Only used if the cache is missing — with Task 3's cache mounted via Input, this
# function is defined but preprocess_all() below should find everything cached
# and never actually call it.
def build_segment_graph(audio_path, segment_duration=SEGMENT_DURATION, sr=SAMPLE_RATE):
    try:
        y, _ = librosa.load(audio_path, sr=sr, duration=10)
    except Exception:
        return None
    if len(y) < sr:
        return None

    hop_length = 512
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=hop_length)
    frames_per_segment = int(segment_duration * sr / hop_length)
    n_segments = max(1, chroma.shape[1] // frames_per_segment)

    node_features = []
    for i in range(n_segments):
        start = i * frames_per_segment
        end = min(start + frames_per_segment, chroma.shape[1])
        node_features.append(chroma[:, start:end].mean(axis=1))

    if len(node_features) < 2:
        return None
    node_features = np.stack(node_features)

    edges = []
    for i in range(len(node_features)):
        if i + 1 < len(node_features):
            edges.append([i, i + 1]); edges.append([i + 1, i])
        for j in range(i + 1, len(node_features)):
            sim = np.dot(node_features[i], node_features[j]) / (
                np.linalg.norm(node_features[i]) * np.linalg.norm(node_features[j]) + 1e-8)
            if sim > SIMILARITY_THRESHOLD:
                edges.append([i, j]); edges.append([j, i])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    x = torch.tensor(node_features, dtype=torch.float)
    return x, edge_index

# %% CELL 7 — Locate the graph cache (reuse Task 3's — no preprocessing here)
GRAPH_CACHE_DIR = TASK3_GRAPH_CACHE if TASK3_GRAPH_CACHE and len(os.listdir(TASK3_GRAPH_CACHE)) > 0 else None

if GRAPH_CACHE_DIR is None:
    print("Task 3 graph cache not found via Input — building it now (this is the "
          "slow path you're trying to avoid; add Task 3's output as an Input to skip this).")
    GRAPH_CACHE_DIR = os.path.join(WORKING_DIR, "task4_graphs_cache")
    os.makedirs(GRAPH_CACHE_DIR, exist_ok=True)
    processed, skipped = 0, 0
    for idx, row in df_matched.iterrows():
        cache_path = os.path.join(GRAPH_CACHE_DIR, f"{idx}.pt")
        if os.path.exists(cache_path):
            continue
        result = build_segment_graph(row['audio_path'])
        if result is None:
            skipped += 1
            continue
        x, edge_index = result
        torch.save({'x': x, 'edge_index': edge_index}, cache_path)
        processed += 1
    print(f"DONE. Processed: {processed}, Skipped: {skipped}")
else:
    print(f"Reusing Task 3's graph cache: {GRAPH_CACHE_DIR} — no preprocessing needed.")

valid_indices = [i for i in range(len(df_matched)) if os.path.exists(os.path.join(GRAPH_CACHE_DIR, f"{i}.pt"))]
df_final = df_matched.iloc[valid_indices].reset_index(drop=True)
graph_files = [os.path.join(GRAPH_CACHE_DIR, f"{i}.pt") for i in valid_indices]
print(f"Final usable samples: {len(df_final)}")

# %% CELL 8 — Train/Val/Test split (same seed as Task 3 -> same split on the same data)
from sklearn.model_selection import train_test_split

indices = list(range(len(df_final)))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# %% CELL 9 — Dataset class (now also carries labels, needed for Cell 16's zero-shot eval)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class RetrievalDataset(TorchDataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        row = df_final.iloc[idx]
        graph_data = torch.load(graph_files[idx], weights_only=False, map_location='cpu')
        encoding = tokenizer(
            row['caption'], truncation=True, padding='max_length',
            max_length=MAX_LEN, return_tensors='pt'
        )
        return {
            'x': graph_data['x'],
            'edge_index': graph_data['edge_index'],
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(row['labels'], dtype=torch.float),
        }

def collate_fn(batch):
    x_list, edge_index_list, batch_idx = [], [], []
    node_offset = 0
    for i, item in enumerate(batch):
        x_list.append(item['x'])
        edge_index_list.append(item['edge_index'] + node_offset)
        batch_idx.extend([i] * item['x'].shape[0])
        node_offset += item['x'].shape[0]
    return {
        'x': torch.cat(x_list, dim=0),
        'edge_index': torch.cat(edge_index_list, dim=1),
        'batch': torch.tensor(batch_idx, dtype=torch.long),
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels': torch.stack([b['labels'] for b in batch]),
    }

# shuffle=False for val/test so embeddings/labels align with df_final order for retrieval eval
train_loader = DataLoader(RetrievalDataset(train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True)
val_loader = DataLoader(RetrievalDataset(val_idx), batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(RetrievalDataset(test_idx), batch_size=BATCH_SIZE, collate_fn=collate_fn)

# %% CELL 10 — Dual Encoder Model
class GNNEncoder(nn.Module):
    def __init__(self, in_channels=N_CHROMA, hidden_channels=64, num_layers=3, out_dim=EMBED_DIM):
        super().__init__()
        self.convs = nn.ModuleList([SAGEConv(in_channels, hidden_channels)])
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
        self.dropout = nn.Dropout(0.3)
        self.proj = nn.Linear(hidden_channels, out_dim)

    def forward(self, x, edge_index, batch):
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = self.dropout(x)
        g = global_mean_pool(x, batch)
        return F.normalize(self.proj(g), dim=-1)


class TextEncoder(nn.Module):
    def __init__(self, out_dim=EMBED_DIM):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.proj = nn.Linear(self.bert.config.hidden_size, out_dim)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return F.normalize(self.proj(cls), dim=-1)


gnn_encoder = GNNEncoder().to(device)
text_encoder = TextEncoder().to(device)

if TASK3_CHECKPOINT is not None:
    task3_ckpt = torch.load(TASK3_CHECKPOINT, weights_only=False, map_location=device)
    gnn_state = {k.replace('gnn.', ''): v for k, v in task3_ckpt['model_state'].items() if k.startswith('gnn.')}
    bert_state = {k.replace('bert.', ''): v for k, v in task3_ckpt['model_state'].items() if k.startswith('bert.')}
    gnn_encoder.load_state_dict(gnn_state, strict=False)
    text_encoder.bert.load_state_dict(bert_state, strict=False)
    print("Warm-started GNN and BERT encoders from Task 3 checkpoint.")
else:
    print("No Task 3 checkpoint found — training both encoders from scratch/pretrained BERT.")

# %% CELL 11 — InfoNCE Contrastive Loss + Training setup
optimizer = torch.optim.AdamW(
    list(gnn_encoder.parameters()) + list(text_encoder.parameters()), lr=LR
)

def info_nce_loss(audio_emb, text_emb, temperature=TEMPERATURE):
    logits = audio_emb @ text_emb.t() / temperature
    labels = torch.arange(logits.shape[0]).to(logits.device)
    loss_a2t = F.cross_entropy(logits, labels)
    loss_t2a = F.cross_entropy(logits.t(), labels)
    return (loss_a2t + loss_t2a) / 2

def train_epoch():
    gnn_encoder.train()
    text_encoder.train()
    total_loss = 0
    for batch in train_loader:
        x = batch['x'].to(device)
        edge_index = batch['edge_index'].to(device)
        b_idx = batch['batch'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        optimizer.zero_grad()
        audio_emb = gnn_encoder(x, edge_index, b_idx)
        text_emb = text_encoder(input_ids, attention_mask)
        loss = info_nce_loss(audio_emb, text_emb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

# %% CELL 12 — Retrieval evaluation (Recall@1/5/10, both directions) + embedding/label extraction
@torch.no_grad()
def get_all_embeddings_and_labels(loader):
    gnn_encoder.eval()
    text_encoder.eval()
    audio_embs, text_embs, labels_list = [], [], []
    for batch in loader:
        x = batch['x'].to(device)
        edge_index = batch['edge_index'].to(device)
        b_idx = batch['batch'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        audio_embs.append(gnn_encoder(x, edge_index, b_idx).cpu())
        text_embs.append(text_encoder(input_ids, attention_mask).cpu())
        labels_list.append(batch['labels'])
    return torch.cat(audio_embs), torch.cat(text_embs), torch.cat(labels_list).numpy()

def compute_recall_at_k(audio_embs, text_embs, k_values=(1, 5, 10)):
    sim = text_embs @ audio_embs.t()
    n = sim.shape[0]
    ranks_t2a = []
    for i in range(n):
        order = torch.argsort(sim[i], descending=True)
        rank = (order == i).nonzero(as_tuple=True)[0].item()
        ranks_t2a.append(rank)
    ranks_t2a = np.array(ranks_t2a)

    sim_a2t = audio_embs @ text_embs.t()
    ranks_a2t = []
    for i in range(n):
        order = torch.argsort(sim_a2t[i], descending=True)
        rank = (order == i).nonzero(as_tuple=True)[0].item()
        ranks_a2t.append(rank)
    ranks_a2t = np.array(ranks_a2t)

    results = {}
    for k in k_values:
        results[f"caption_to_audio_R@{k}"] = float((ranks_t2a < k).mean())
        results[f"audio_to_caption_R@{k}"] = float((ranks_a2t < k).mean())
    return results

# %% CELL 13 — Resume + Training loop
start_epoch = 0
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, weights_only=False, map_location=device)
    gnn_encoder.load_state_dict(ckpt['gnn_state'])
    text_encoder.load_state_dict(ckpt['text_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch = ckpt['epoch'] + 1
    print(f"Resumed from epoch {start_epoch}")

for epoch in range(start_epoch, NUM_EPOCHS):
    train_loss = train_epoch()
    val_audio_embs, val_text_embs, _ = get_all_embeddings_and_labels(val_loader)
    val_recall = compute_recall_at_k(val_audio_embs, val_text_embs)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | "
          f"Val C2A R@10: {val_recall['caption_to_audio_R@10']:.4f} | "
          f"Val A2C R@10: {val_recall['audio_to_caption_R@10']:.4f}")

    torch.save({
        'epoch': epoch,
        'gnn_state': gnn_encoder.state_dict(),
        'text_state': text_encoder.state_dict(),
        'optimizer_state': optimizer.state_dict(),
    }, CHECKPOINT_PATH)

# %% CELL 14 — Final Test Evaluation (spec deliverable #2: retrieval table)
test_audio_embs, test_text_embs, test_labels = get_all_embeddings_and_labels(test_loader)
test_recall = compute_recall_at_k(test_audio_embs, test_text_embs)

print(f"\n{'='*55}\nTASK 4 — RETRIEVAL EVALUATION TABLE (MusicCaps test split)\n{'='*55}")
print(f"{'Metric':<25}{'Value':<10}")
for k, v in test_recall.items():
    print(f"{k:<25}{v:<10.4f}")

# %% CELL 15 — 10 qualitative retrieval examples (spec deliverable #3)
test_df_final = df_final.iloc[test_idx].reset_index(drop=True)
sim = test_text_embs @ test_audio_embs.t()

qualitative_examples = []
print("\n--- Qualitative Retrieval Examples (caption -> top-3 audio matches) ---")
np.random.seed(7)
sample_query_indices = np.random.choice(len(test_idx), min(10, len(test_idx)), replace=False)
for qi in sample_query_indices:
    top3 = torch.argsort(sim[qi], descending=True)[:3].tolist()
    query_caption = test_df_final.iloc[qi]['caption']
    print(f"\nQuery caption: {query_caption[:100]}...")
    matches = []
    for rank, ai in enumerate(top3):
        is_correct = (ai == qi)
        fname = os.path.basename(test_df_final.iloc[ai]['audio_path'])
        marker = " <-- CORRECT MATCH" if is_correct else ""
        print(f"  Top-{rank+1}: {fname}{marker}")
        matches.append({"rank": rank + 1, "audio_file": fname, "is_correct_match": bool(is_correct)})
    qualitative_examples.append({"query_caption": query_caption, "top3_matches": matches})

# %% CELL 16 — NEW: Zero-shot tag prediction from captions vs. Task 3 supervised model
# spec deliverable #4. Uses the CONTRASTIVELY-TRAINED text encoder to embed each of
# the 50 tag names, then classifies each test clip's audio embedding by cosine
# similarity to those tag embeddings — no labeled fine-tuning for this task, hence
# "zero-shot". The threshold is tuned on the val split (to pick a fair operating
# point) and then applied once to test. Compared directly against Task 3's
# supervised cross-attention model's saved scores.
tag_texts = [f"the music has the following quality: {tag}" for tag in TAG_NAMES]
tag_encodings = tokenizer(tag_texts, truncation=True, padding='max_length',
                           max_length=32, return_tensors='pt')
with torch.no_grad():
    text_encoder.eval()
    tag_embeddings = text_encoder(
        tag_encodings['input_ids'].to(device), tag_encodings['attention_mask'].to(device)
    )  # (NUM_TAGS, EMBED_DIM), L2-normalized

@torch.no_grad()
def zero_shot_scores(audio_embs):
    # audio_embs: (N, EMBED_DIM), L2-normalized. tag_embeddings: (NUM_TAGS, EMBED_DIM).
    return (audio_embs.to(device) @ tag_embeddings.t()).cpu().numpy()  # (N, NUM_TAGS) cosine sim

val_audio_embs, _, val_labels = get_all_embeddings_and_labels(val_loader)
val_scores = zero_shot_scores(val_audio_embs)
test_scores = zero_shot_scores(test_audio_embs)

# Tune a single global threshold on val to maximize macro-F1
best_threshold, best_val_f1 = 0.0, -1.0
for t in np.linspace(val_scores.min(), val_scores.max(), 50):
    preds = (val_scores > t).astype(np.float32)
    f1 = f1_score(val_labels, preds, average='macro', zero_division=0)
    if f1 > best_val_f1:
        best_val_f1, best_threshold = f1, t
print(f"\nZero-shot threshold tuned on val: {best_threshold:.4f} (val Macro-F1: {best_val_f1:.4f})")

test_preds = (test_scores > best_threshold).astype(np.float32)
zs_macro_f1 = f1_score(test_labels, test_preds, average='macro', zero_division=0)
zs_micro_f1 = f1_score(test_labels, test_preds, average='micro', zero_division=0)
valid_tags = [k for k in range(test_labels.shape[1]) if test_labels[:, k].sum() > 0]
zs_auc_pr = (average_precision_score(test_labels[:, valid_tags], test_scores[:, valid_tags], average='macro')
             if valid_tags else float('nan'))

zero_shot_results = {"macro_f1": zs_macro_f1, "micro_f1": zs_micro_f1, "auc_pr": zs_auc_pr,
                      "threshold": float(best_threshold)}

print(f"\n{'='*65}\nTASK 4 ZERO-SHOT vs. TASK 3 SUPERVISED (spec deliverable #4)\n{'='*65}")
print(f"{'Model':<35}{'Macro-F1':<12}{'Micro-F1':<12}{'AUC-PR':<12}")
print(f"{'Task 4: Zero-shot (contrastive)':<35}{zs_macro_f1:<12.4f}{zs_micro_f1:<12.4f}{zs_auc_pr:<12.4f}")
if task3_results is not None and 'cross_attention' in task3_results.get('ablation_results', {}):
    t3 = task3_results['ablation_results']['cross_attention']
    print(f"{'Task 3: Supervised (cross-attn)':<35}{t3['macro_f1']:<12.4f}{t3['micro_f1']:<12.4f}{t3['auc_pr']:<12.4f}")
else:
    print("Task 3 supervised scores not available (task3_results.json not found or "
          "missing 'cross_attention' entry — add it as an Input to get this comparison).")

# %% CELL 17 — NEW: Human evaluation template (spec Section 6 requirement)
# Exports the 10 qualitative examples above as a CSV with blank rating columns
# for 5 listeners to fill in by hand (scale 1-5: does the retrieved clip match
# the caption?). This can't be automated, so this cell just prepares the sheet.
human_eval_rows = []
for ex in qualitative_examples:
    for m in ex['top3_matches']:
        human_eval_rows.append({
            "query_caption": ex['query_caption'],
            "rank": m['rank'],
            "audio_file": m['audio_file'],
            "listener_1_rating": "", "listener_2_rating": "", "listener_3_rating": "",
            "listener_4_rating": "", "listener_5_rating": "",
        })
human_eval_df = pd.DataFrame(human_eval_rows)
HUMAN_EVAL_PATH = os.path.join(WORKING_DIR, "task4_human_eval_template.csv")
human_eval_df.to_csv(HUMAN_EVAL_PATH, index=False)
print(f"Human evaluation template saved to: {HUMAN_EVAL_PATH}")
print("Have 5 listeners rate each retrieved clip 1-5 for caption match, then average per query.")

# %% CELL 18 — Save results
RESULTS_PATH = os.path.join(WORKING_DIR, "task4_results.json")
results = {
    "test_recall": test_recall,
    "zero_shot_vs_task3": {
        "task4_zero_shot": zero_shot_results,
        "task3_supervised": task3_results['ablation_results']['cross_attention'] if task3_results else None,
    },
    "qualitative_examples": qualitative_examples,
    "num_samples": len(df_final),
    "embed_dim": EMBED_DIM,
    "temperature": TEMPERATURE,
}
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"Checkpoint saved at: {CHECKPOINT_PATH}")
print(f"Results saved at: {RESULTS_PATH}")
print(f"Human eval template saved at: {HUMAN_EVAL_PATH}")
print("\nIMPORTANT: click 'Save Version' (top right) now, or these files")
print("will be lost when this session ends.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.4 MB/s eta 0:00:0000:0100:01
CSV_PATH: /kaggle/input/datasets/googleai/musiccaps/musiccaps-public.csv
AUDIO_DIR: /kaggle/input/datasets/ayushsharma4045/musiccapsaudio5308/audioFiles/audioFiles
Task 3 checkpoint (warm-start GNN+BERT): None
Task 3 graph cache (skip preprocessing): None
Task 3 results.json (tag vocab + supervised scores): None
Using device: cuda
Matched 5308 / 5521 captions to audio files
Task 3 graph cache not found via Input — building it now (this is the slow path you're trying to avoid; add Task 3's output as an Input to skip this).


/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


DONE. Processed: 5308, Skipped: 0
Final usable samples: 5308
Train: 4246, Val: 531, Test: 531


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


No Task 3 checkpoint found — training both encoders from scratch/pretrained BERT.
Epoch 1/20 | Loss: 3.3616 | Val C2A R@10: 0.0358 | Val A2C R@10: 0.0471
Epoch 2/20 | Loss: 3.2360 | Val C2A R@10: 0.0414 | Val A2C R@10: 0.0659
Epoch 3/20 | Loss: 3.1728 | Val C2A R@10: 0.0414 | Val A2C R@10: 0.0584
Epoch 4/20 | Loss: 3.1028 | Val C2A R@10: 0.0546 | Val A2C R@10: 0.0791
Epoch 5/20 | Loss: 3.0350 | Val C2A R@10: 0.0640 | Val A2C R@10: 0.0640
Epoch 6/20 | Loss: 2.9728 | Val C2A R@10: 0.0678 | Val A2C R@10: 0.0621
Epoch 7/20 | Loss: 2.9071 | Val C2A R@10: 0.0659 | Val A2C R@10: 0.0716
Epoch 8/20 | Loss: 2.8460 | Val C2A R@10: 0.0847 | Val A2C R@10: 0.0753
Epoch 9/20 | Loss: 2.7663 | Val C2A R@10: 0.0678 | Val A2C R@10: 0.0565
Epoch 10/20 | Loss: 2.6966 | Val C2A R@10: 0.0716 | Val A2C R@10: 0.0772
Epoch 11/20 | Loss: 2.6414 | Val C2A R@10: 0.0772 | Val A2C R@10: 0.0640
Epoch 12/20 | Loss: 2.5606 | Val C2A R@10: 0.0847 | Val A2C R@10: 0.0885
Epoch 13/20 | Loss: 2.5006 | Val C2A R@10: 0.0753 |